# Notebook 02 — Medallion Architecture Exploration
**Purpose:** Validate Bronze/Silver/Gold design decisions by exploring what each layer contains and why.
Run after 01_bronze_ingestion.py and 02_silver_cleaning.py have executed in Databricks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

SAMPLE_DIR = Path('../data/sample')
print('Setup complete.')

## 1. Bronze Layer — Raw Data As-Is
Bronze is append-only. No transformations. Every API response lands here unchanged.
This is the immutable audit trail of what arrived and when.

In [ ]:
# Load sample data representing Bronze layer content
df_products = pd.read_csv(SAMPLE_DIR / 'products_sample.csv')
df_orders   = pd.read_csv(SAMPLE_DIR / 'orders_sample.csv')
df_events   = pd.read_csv(SAMPLE_DIR / 'events_sample.csv')

print('=== Bronze Layer — Raw Shapes ===')
print(f'products: {df_products.shape}')
print(f'orders:   {df_orders.shape}')
print(f'events:   {df_events.shape}')
df_products.head(3)

## 2. Silver Layer — Cleaned and Enriched
Silver cleans nulls, deduplicates via MERGE, flattens nested structs, and enriches records.
One row per entity. No aggregation yet.

In [ ]:
# Simulate Silver cleaning on the sample data
df_silver_products = df_products.copy()

# Flatten rating struct (done in 02_silver_cleaning.py)
if 'rating' in df_silver_products.columns:
    import ast
    df_silver_products['rating_rate']  = df_silver_products['rating'].apply(
        lambda x: ast.literal_eval(x)['rate'] if isinstance(x, str) else x.get('rate') if isinstance(x, dict) else None)
    df_silver_products['rating_count'] = df_silver_products['rating'].apply(
        lambda x: ast.literal_eval(x)['count'] if isinstance(x, str) else x.get('count') if isinstance(x, dict) else None)
    df_silver_products = df_silver_products.drop(columns=['rating'])

# Standardize category to lowercase
df_silver_products['category'] = df_silver_products['category'].str.lower().str.strip()

print('=== Silver Layer — Cleaned Products ===')
print(df_silver_products.dtypes)
print(f'\nNull counts:\n{df_silver_products.isnull().sum()}')
df_silver_products.head(3)

## 3. Gold Layer — Customer KPIs
Gold aggregates Silver into one row per customer with business-ready metrics.
This is what dbt reads from BigQuery.

In [ ]:
import numpy as np

# Simulate Gold aggregation
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

df_gold = df_orders.groupby('user_id').agg(
    total_orders    = ('cart_id', 'count'),
    total_spend     = ('order_value', 'sum'),
    avg_order_value = ('order_value', 'mean'),
    first_order_date = ('order_date', 'min'),
    last_order_date  = ('order_date', 'max')
).reset_index()

df_gold['customer_segment'] = pd.cut(
    df_gold['total_spend'],
    bins=[0, 200, 500, float('inf')],
    labels=['low_value', 'mid_value', 'high_value']
)

print('=== Gold Layer — Customer KPIs ===')
print(df_gold.describe().round(2))
display(df_gold)

## 4. Layer Comparison
Visualize what each layer contributes and why the medallion architecture is worth the complexity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Medallion Architecture — Layer Comparison', fontsize=14, fontweight='bold')

# Bronze: raw price distribution
axes[0].hist(df_products['price'], bins=10, color='#CD7F32', edgecolor='white')
axes[0].set_title('Bronze — Raw Prices')
axes[0].set_xlabel('Price ($)')

# Silver: cleaned categories
cat_counts = df_silver_products['category'].value_counts()
axes[1].barh(cat_counts.index, cat_counts.values, color='#C0C0C0')
axes[1].set_title('Silver — Category Distribution')

# Gold: customer segments
seg_counts = df_gold['customer_segment'].value_counts()
axes[2].pie(seg_counts.values, labels=seg_counts.index,
            autopct='%1.0f%%', colors=['#FFD700', '#C0C0C0', '#CD7F32'])
axes[2].set_title('Gold — Customer Segments')

plt.tight_layout()
plt.savefig('../data/sample/medallion_layers.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nMedallion Architecture Summary:')
print('  Bronze: Raw API data, append-only, no transformations')
print('  Silver: Cleaned, deduplicated, enriched via MERGE')
print('  Gold:   Aggregated KPIs, business-ready, exported to BigQuery')